In [ ]:
#1.1 Finding where the leaks originate from
import numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from statsmodels.stats.proportion import proportion_confint

# ---------- 1. LOAD AND DEFINE VARIABLES ----------
df = pd.read_excel("DataSets/Competition_Dataset_Final.xlsx")
df["leak"] = (df["Failure Category"] == "Leak").astype(int)       # outcome
df["F2"]   = (df["Facility"] == "Facility 2").astype(int)          # predictor 1
df["SB"]   = (df["Coil Supplier"] == "Supplier B").astype(int)     # predictor 2
df["F2xSB"]= df["F2"] * df["SB"]

# ---------- 2. THE 2x2 TABLE WITH EXACT INTERVALS ----------
cells = df.groupby(["F2", "SB"])["leak"].agg(LeakedUnits="sum", units="count")
cells["rate"] = cells.LeakedUnits / cells.units
ci = [proportion_confint(k, units, method="beta") for LeakedUnits, units in zip(cells.LeakedUnits, cells.units)]  # Clopper-Pearson: safe at 0
cells["lo"], cells["hi"] = [c[0] for c in ci], [c[1] for c in ci]
print(cells.round(3), "\n")

# ---------- 3. CANDIDATE MODELS (each: fit on train -> predict on test) ----------
EPS = 1e-3
def fit_null(tr, te):
    return np.full(len(te), tr.leak.mean())

def _logit(cols):
    def f(tr, te):
        m = LogisticRegression(C=1.0, max_iter=1000).fit(tr[cols], tr.leak)   # L2 penalty handles separation
        return m.predict_proba(te[cols])[:, 1]
    return f
fit_additive    = _logit(["F2", "SB"])
fit_interaction = _logit(["F2", "SB", "F2xSB"])

def fit_either(tr, te):   # one switch: risk = F2 OR SB; rate per group (lightly smoothed)
    r = ((tr.F2 | tr.SB) == 1)
    p1 = (tr.leak[r].sum() + .5) / (r.sum() + 1); p0 = (tr.leak[~r].sum() + .5) / ((~r).sum() + 1)
    return np.where((te.F2 | te.SB) == 1, p1, p0)

def _or_pred(theta, F2, SB):
    a, b, c = 1 / (1 + np.exp(-theta))           # base, F2 effect, SB effect (all in 0-1)
    return 1 - (1 - a) * (1 - b * F2) * (1 - c * SB)
def fit_noisyor(tr, te):
    def nll(t):
        p = np.clip(_or_pred(t, tr.F2.values, tr.SB.values), 1e-9, 1 - 1e-9)
        return -(tr.leak.values * np.log(p) + (1 - tr.leak.values) * np.log(1 - p)).sum()
    th = minimize(nll, [-3, 0, 0], method="Nelder-Mead", options={"maxiter": 4000}).x
    return _or_pred(th, te.F2.values, te.SB.values)

models = {"null": fit_null, "additive": fit_additive, "noisy-OR": fit_noisyor,
          "either-factor switch": fit_either, "interaction": fit_interaction}

# ---------- 4. REPEATED CROSS-VALIDATION ----------
def logloss(y, p):
    p = np.clip(p, EPS, 1 - EPS); return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
strat = df.F2.astype(str) + df.SB.astype(str)          # stratify on the 4 cells so every fold has all of them
scores = {k: [] for k in models}
for tr_i, te_i in cv.split(df, strat):
    tr, te = df.iloc[tr_i], df.iloc[te_i]
    for name, fn in models.items():
        scores[name].append(logloss(te.leak.values, fn(tr, te)))
print(pd.DataFrame({k: [np.mean(v), np.std(v)] for k, v in scores.items()}, index=["mean", "sd"]).T.round(4), "\n")

# ---------- 5. ARE THE TWO FACTORS INDEPENDENT? (bootstrap test on the 'both' cell) ----------
rng = np.random.default_rng(0)
def or_both(d):
    a = d[(d.F2 == 1) & (d.SB == 0)].leak.mean(); b = d[(d.F2 == 0) & (d.SB == 1)].leak.mean()
    return 1 - (1 - a) * (1 - b)          # what independence would predict for units with both
gaps = []
for _ in range(2000):
    s = df.sample(len(df), replace=True, random_state=int(rng.integers(1e9)))
    gaps.append(s[(s.F2 == 1) & (s.SB == 1)].leak.mean() - or_both(s))
obs = df[(df.F2 == 1) & (df.SB == 1)].leak.mean()
print(f"observed 'both' rate {obs:.3f} | independence predicts {or_both(df):.3f} | "
      f"gap 95% CI {np.percentile(gaps, [2.5, 97.5]).round(3)}\n")

# ---------- 6. DOES ANYTHING ELSE MATTER INSIDE THE RISK GROUP? ----------
risk = df[(df.F2 | df.SB) == 1]
for v in ["Shift", "Product Family", "Customer Region", "Controls Revision", "Facility", "Coil Supplier"]:
    t = pd.crosstab(risk[v], risk.leak); print(f"{v:18s} chi2 p = {chi2_contingency(t)[1]:.3f}")

         k  units   rate     lo     hi
F2 SB                                 
0  0     0   1570  0.000  0.000  0.002
   1   356    646  0.551  0.512  0.590
1  0   290    536  0.541  0.498  0.584
   1   142    248  0.573  0.508  0.635 

                        mean      sd
null                  0.5760  0.0106
additive              0.4005  0.0108
noisy-OR              0.3367  0.0052
either-factor switch  0.3287  0.0021
interaction           0.3347  0.0020 

observed 'both' rate 0.573 | independence predicts 0.794 | gap 95% CI [-0.292 -0.156]

Shift              chi2 p = 0.569
Product Family     chi2 p = 0.477
Customer Region    chi2 p = 0.203
Controls Revision  chi2 p = 0.339
Facility           chi2 p = 0.868
Coil Supplier      chi2 p = 0.177


MEANING
2x2 Table:
F2 = unit came from factory 2
SB = unit fitted with supplier B coils
Leaked Units = number of units in the population that had a leak
Units = number of total units
Rate = how many units leaked
Lo Hi = 95% confidence interval
First row shows that if the unit wasn't built in Factory 2 AND didn't use supplier B coils, it didn't have a leak issue
Other three rows all overlap with rates between 54% and 57%. Since they are all so similar the best explanation/model is the Either factor switch (A logistic regression (or Bernoulli model) with one binary indicator for “Facility 2 OR Supplier B.). This is because there are two risk factors but when both risk factors are present, the rate doesn't increase. Bottom line is, Among failing units, about 55% of those built at Facility 2 or fitted with Supplier B coils failed from a leak, compared with 0% (0 of 1,570) of units with neither.

Candidate Table
The lower the mean, the more accurate the model fits the data. it shows that the either-factor switch is the best model. Interaction is also a very good model however it has extra noise that reduces its accuracy. 

Independence Statement
If they were independent the rate would be around .794 but since it stays at 0.573 when both cases are true, the are dependent

Other variabel dependence
Since all p values are above 0.17, it shows that the leaks aren't majorly affect by customer region, product family, shift, or other variables. 